# Book Price Data Collection

This notebook demonstrates the web-scraping stage of the **Price & Product Intelligence Tracker** project.

### Objective
Collect book-level product information from Books to Scrape, including:
- Book title
- Price
- Availability
- Star rating

The notebook first tests the extraction logic on one catalogue page and then scales the process to all 50 catalogue pages.

> **Note:** `scraper.py` contains the more automated version used for repeated tracking runs, including a scrape date and append logic. This notebook is retained as the exploratory/testing record.

In [1]:
# Step 1: Request a catalogue page and parse its HTML structure.
import requests
import time
from bs4 import BeautifulSoup

url = "http://books.toscrape.com/catalogue/page-1.html"
response = requests.get(url)
response.encoding = 'utf-8'
soup = BeautifulSoup(response.text, "html.parser")

/Users/niharnandanwar/myenv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
# Step 2: Locate each book card and extract the required product attributes.
books = soup.find_all("article", class_="product_pod")

for book in books:
    title = book.h3.a["title"]
    price = book.find("p", class_="price_color").text
    availability = book.find("p", class_="instock availability").text.strip()
    rating = book.p["class"][1]
    print(title, price, availability, rating)

A Light in the Attic £51.77 In stock Three
Tipping the Velvet £53.74 In stock One
Soumission £50.10 In stock One
Sharp Objects £47.82 In stock Four
Sapiens: A Brief History of Humankind £54.23 In stock Five
The Requiem Red £22.65 In stock One
The Dirty Little Secrets of Getting Your Dream Job £33.34 In stock Four
The Coming Woman: A Novel Based on the Life of the Infamous Feminist, Victoria Woodhull £17.93 In stock Three
The Boys in the Boat: Nine Americans and Their Epic Quest for Gold at the 1936 Berlin Olympics £22.60 In stock Four
The Black Maria £52.15 In stock One
Starving Hearts (Triangular Trade Trilogy, #1) £13.99 In stock Two
Shakespeare's Sonnets £20.66 In stock Four
Set Me Free £17.46 In stock Five
Scott Pilgrim's Precious Little Life (Scott Pilgrim #1) £52.29 In stock Five
Rip it Up and Start Again £35.02 In stock Five
Our Band Could Be Your Life: Scenes from the American Indie Underground, 1981-1991 £57.25 In stock Three
Olio £23.88 In stock One
Mesaerion: The Best Scienc

In [3]:
# Step 3: Store the extracted product records as dictionaries.
data = []

for book in books:
    title = book.h3.a["title"]
    price = book.find("p", class_="price_color").text
    availability = book.find("p", class_="instock availability").text.strip()
    rating = book.p["class"][1]
    data.append({"title": title, "price": price, "availability": availability, "rating": rating})

In [4]:
# Step 4: Convert the records into a DataFrame and save the single-page test dataset.
import pandas as pd

df = pd.DataFrame(data)
df.to_csv("books_page1.csv", index=False)
df

,title,price,availability,rating
0,A Light in the Attic,£51.77,In stock,Three
1,Tipping the Velvet,£53.74,In stock,One
2,Soumission,£50.10,In stock,One
3,Sharp Objects,£47.82,In stock,Four
4,Sapiens: A Brief History of Humankind,£54.23,In stock,Five
5,The Requiem Red,£22.65,In stock,One
6,The Dirty Little Secrets of Getting Your Dream...,£33.34,In stock,Four
7,The Coming Woman: A Novel Based on the Life of...,£17.93,In stock,Three
8,The Boys in the Boat: Nine Americans and Their...,£22.60,In stock,Four
9,The Black Maria,£52.15,In stock,One


In [5]:
# Step 5: Apply the tested extraction logic across all 50 catalogue pages and save the full dataset.
all_data = []

for page in range(1, 51):
    url = f"http://books.toscrape.com/catalogue/page-{page}.html"
    resposne = requests.get(url)
    response.encoding = 'utf-8'
    soup = BeautifulSoup(response.text, "html.parser")
    books = soup.find_all("article", class_="product_pod")

    for book in books:
        title = book.h3.a["title"]
        price = book.find("p", class_="price_color").text
        availability = book.find("p", class_="instock availability").text.strip()
        rating = book.p["class"][1]
        all_data.append({"title":title, "price":price, "availability":availability, "rating":rating})

    time.sleep(1)

df_all = pd.DataFrame(all_data)
df_all.to_csv("all_books.csv", index = False)
df_all

,title,price,availability,rating
0,A Light in the Attic,£51.77,In stock,Three
1,Tipping the Velvet,£53.74,In stock,One
2,Soumission,£50.10,In stock,One
3,Sharp Objects,£47.82,In stock,Four
4,Sapiens: A Brief History of Humankind,£54.23,In stock,Five
...,...,...,...,...
995,Our Band Could Be Your Life: Scenes from the A...,£57.25,In stock,Three
996,Olio,£23.88,In stock,One
997,Mesaerion: The Best Science Fiction Stories 18...,£37.59,In stock,One
998,Libertarianism for Beginners,£51.33,In stock,Two


In [6]:
# Dedeplication code to be ran after manual testing
# import pandas as pd

# df = pd.read_csv("data/all_books.csv")
# df = df.drop_duplicates(subset=["title", "scrape_date"], keep="first")
# df.to_csv("data/all_books.csv", index=False)
# print(df.shape)